# Understand the baseline

Goal: create a small Core Building and explicitly assign a construction set from Data. Run from this examples folder with the installed package and its examples extra. Inputs are teaching assumptions, not a surveyed building. No engine or network is used.

In [ ]:
from epsm_core import Building, FacadeSpec, Footprint, GlazingSpec
from epsm_data import reference_construction_set
from epsm_data.domain import BuildingElement as Role

from epsm_lca import AssessmentSettings, assess_renovation

constructions = reference_construction_set()
window = constructions.elements[Role.WINDOW]
facade = FacadeSpec(
    default_wwr=0.2,
    glazing=GlazingSpec(
        u_value_w_m2k=window.u_value,
        solar_heat_gain_coefficient=window.shgc,
        visible_transmittance=window.vt,
    ),
)
baseline = Building.from_footprint(
    Footprint.rectangle(5, 5),
    storeys=1,
    storey_height_m=2.5,
    construction_set=constructions,
    facade=facade,
)
print(baseline)

The outline is 5 m by 5 m and one storey is 2.5 m tall. Outside walls have 50 m² gross area. Windows occupy 20%, leaving **10 m² of windows and 40 m² of solid wall**. Geometry gives area; unknown window mass remains unknown.

In [ ]:
import math

wall_area = sum(e.area_m2 for e in baseline.elements if e.role == Role.EXT_WALL)
window_area = sum(e.area_m2 for e in baseline.elements if e.role == Role.WINDOW)
assert math.isclose(wall_area, 40)
assert math.isclose(window_area, 10)
print("Wall area:", wall_area, "m²; window area:", window_area, "m²")
print(baseline.quantities().by_material())

Existing materials are retained by default. With no renovation and energy excluded, new material impact is zero. This does **not** claim that original construction had zero impact.

In [ ]:
settings = AssessmentSettings(30, include_energy=False)
report = assess_renovation(baseline, settings)
assert report.complete and report.baseline.total_kgco2e == 0
print(report.summary())

Climate factors describe impact per kg, m², m³ or item. The teaching construction intentionally has no environmental factors. Notebook 5 shows how to use the actual Boverket values already held by Data.